# **DATA PREPROCESSING TABULAR**

---

## **1. Import Libraries**

In [1]:
import pandas as pd
import numpy as np
import scipy.stats as stats

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.metrics import mean_squared_error

## **2. Load Raw Data For Preprocessing**

Dựa trên những phát hiện quan trọng thu được ở bước khám phá dữ liệu (EDA) — giai đoạn tiền xử lý được triển khai nhằm chuẩn hóa cấu trúc dữ liệu, đảm bảo mỗi biến phản ánh đúng bản chất định tính hoặc định lượng của nó. 

Khác với các bộ dữ liệu đã được làm sạch sẵn, dữ liệu điều tra dân số thực tế thường chứa các ký tự dị thường (`?`). Bước khởi tạo này sẽ nạp dữ liệu và ép kiểu các ký tự lỗi về chuẩn `np.nan` của thư viện Pandas.

In [2]:
adult_path = "../data/raw/adult.csv"
df_adult = pd.read_csv(adult_path)
df_adult = df_adult.replace(regex=r'^\s*\?\s*$', value=np.nan)

num_cols = ['age', 'fnlwgt', 'education.num', 'capital.gain', 'capital.loss', 'hours.per.week']
for col in num_cols:
    if col in df_adult.columns:
        df_adult[col] = pd.to_numeric(df_adult[col], errors='coerce')

Dữ liệu được đọc từ tệp `adult.csv` thành công và được khởi tạo dưới dạng cấu trúc dữ liệu bảng (tabular), lưu trữ trong biến DataFrame `df_adult`.

## **3. Controlled Missing Value Imputation**

**Mục tiêu:** Thay vì áp dụng mù quáng một thuật toán điền khuyết, nhóm tiến hành đánh giá định lượng hiệu quả của 5 chiến lược thông qua môi trường mô phỏng (simulation). Cụ thể, nhóm sẽ giả lập mất mát dữ liệu (MCAR) trên một biến số học hoàn chỉnh, sau đó dùng các thuật toán để nội suy và tính toán sai số RMSE so với Ground Truth.

### **3.1 Theoretical Foundation & Mathematical Formulas**

Trước khi thực thi, việc thấu hiểu nền tảng toán học và giả định của từng thuật toán là bắt buộc để đánh giá rủi ro sai lệch (bias) mà chúng có thể mang lại.

#### 1. Arithmetic Mean Imputation 

* **Lý thuyết:** Đây là kỹ thuật điền khuyết đơn giản nhất dành cho dữ liệu liên tục. Nó sử dụng thước đo xu hướng tập trung, giả định rằng giá trị trung bình là giá trị có khả năng xuất hiện cao nhất đối với một quan sát lấy ngẫu nhiên từ phân phối chuẩn (Gaussian). Tuy nhiên, nó cực kỳ nhạy cảm với ngoại lai, làm giảm nhân tạo phương sai của biến và thu hẹp khoảng tin cậy.
* **Toán học:** Giá trị trung bình mẫu $\bar{x}$ được tính bằng:
  $$\bar{x} = \frac{\sum_{i=1}^{n} x_i}{n}$$

#### 2. Median Imputation

* **Lý thuyết:** Một giải pháp thay thế mạnh mẽ cho phương pháp trung bình, đặc biệt khi dữ liệu có độ lệch (skewness) cao hoặc chứa ngoại lai cực đoan (như biến `capital.gain` trong tập dữ liệu này). Tuy nhiên, nó vẫn làm biến dạng phân phối do tạo ra một spike nhân tạo tại vùng trung tâm.
* **Toán học:** Dữ liệu quan sát được sắp xếp tăng dần: $x_{(1)}, x_{(2)}, \dots, x_{(n)}$.
  * Nếu $n$ lẻ: Trung vị là $x_{(\frac{n+1}{2})}$.
  * Nếu $n$ chẵn: Trung vị là $\frac{x_{(\frac{n}{2})} + x_{(\frac{n}{2} + 1)}}{2}$.

#### 3. Mode Imputation

* **Lý thuyết:** Giá trị xuất hiện với tần suất cao nhất. Chủ yếu dùng cho biến phân loại (Categorical). Mặc dù đơn giản, phương pháp này thất bại nếu phân phối đa đỉnh (multimodal) và bỏ qua hoàn toàn mối tương quan giữa các đặc trưng.
* **Toán học:** Là giá trị $x_i$ cực đại hóa hàm tần suất $f(x)$.

#### 4. k-Nearest Neighbors (k-NN) Imputation

* **Lý thuyết:** k-NN là thuật toán lazy learning phi tham số. Thay vì dùng một hằng số toàn cục, nó nội suy bằng cách tìm $k$ quan sát hoàn chỉnh giống nhất (hàng xóm) trong không gian đặc trưng.
  * $k=3$: Bám sát mẫu cục bộ, rủi ro quá khớp (overfitting) với nhiễu.
  * $k=5$: Trạng thái cân bằng tối ưu, lọc được nhiễu nhưng vẫn giữ được cấu trúc.
  * $k=10$: Rủi ro trơn hóa quá mức (underfitting), dần hội tụ về trung bình toàn cục.
* **Toán học:** Tính khoảng cách (ví dụ: Euclidean) giữa các điểm dữ liệu:
  $$d(a, b) = \sqrt{\sum_{i=1}^{m} (a_i - b_i)^2}$$

#### 5. Multiple Imputation by Chained Equations (MICE)

* **Lý thuyết:** Còn gọi là Fully Conditional Specification (FCS). Hoạt động dưới giả định MAR, MICE lập mô hình từng biến bị khuyết như một hàm của các biến còn lại thông qua một chuỗi các phương trình hồi quy. MICE bảo toàn hình dáng phân phối và tính đa đỉnh (multimodality) xuất sắc.
* **Toán học:** Mô hình hồi quy cơ bản cho biến liên tục:
  $$x_{j, obs} = \beta_0 + \mathbf{X}_{-j, obs} \boldsymbol{\beta} + \epsilon_j, \quad \epsilon_j \sim N(0, \sigma^2)$$
  *Luật kết hợp Rubin (Rubin's Rules):* Tổng hợp từ $H$ bộ dữ liệu với phương sai tổng cộng $T$:
  $$T = \bar{V}_H + \left(1 + \frac{1}{H}\right) B_H$$

### **3.2 MCAR Injection**

Nhóm lựa chọn thuộc tính `age` (Tuổi) làm đối tượng thử nghiệm vì cột này nguyên bản không có giá trị khuyết (hoàn hảo để làm Ground Truth) và có tương quan với nhiều biến kinh tế khác. Nhóm sẽ chủ động xóa ngẫu nhiên 10% (MCAR) dữ liệu tại cột này.

In [3]:
# Define target and ground truth
target_col = 'age'
ground_truth = df_adult[target_col].copy()

# Simulate missing data (10%)
df_sim = df_adult.copy()
np.random.seed(42)
drop_ratio = 0.10
n_rows = len(df_sim)
mask_missing = np.random.rand(n_rows) < drop_ratio
df_sim.loc[mask_missing, target_col] = np.nan

print("\tMISSING DATA SIMULATION")
print(f"Missing values introduced: {mask_missing.sum():,}")
print(f"Missing ratio            : {(mask_missing.sum() / n_rows) * 100:.2f}%")

missing_indices = df_sim[df_sim[target_col].isna()].index

	MISSING DATA SIMULATION
Missing values introduced: 3,292
Missing ratio            : 10.11%


### **3.3 Implementation & Evaluation**

Nhóm tiến hành cài đặt 5 chiến lược (kèm các biến thể của k-NN) và đo lường *RMSE (Root Mean Square Error)*. Chỉ số RMSE càng thấp chứng tỏ giá trị nội suy càng bám sát với dữ liệu thực tế (Ground Truth) đã bị xóa.

In [4]:
# Prepare numerical data
df_num_sim = df_sim[num_cols].copy()
true_values = ground_truth.loc[missing_indices]
def compute_rmse(y_true, y_pred):
    return mean_squared_error(y_true, y_pred, squared=False)

# Evaluate imputation strategies
results = {}

# Mean
imputer = SimpleImputer(strategy='mean')
df_imputed = pd.DataFrame(imputer.fit_transform(df_num_sim), columns=num_cols)
results['Mean'] = compute_rmse(true_values, df_imputed.loc[missing_indices, target_col])

# Median
imputer = SimpleImputer(strategy='median')
df_imputed = pd.DataFrame(imputer.fit_transform(df_num_sim), columns=num_cols)
results['Median'] = compute_rmse(true_values, df_imputed.loc[missing_indices, target_col])

# Mode
imputer = SimpleImputer(strategy='most_frequent')
df_imputed = pd.DataFrame(imputer.fit_transform(df_num_sim), columns=num_cols)
results['Mode'] = compute_rmse(true_values, df_imputed.loc[missing_indices, target_col])

# k-NN (k = 3, 5, 10)
for k in [3, 5, 10]:
    imputer = KNNImputer(n_neighbors=k)
    df_imputed = pd.DataFrame(imputer.fit_transform(df_num_sim), columns=num_cols)
    results[f'KNN (k={k})'] = compute_rmse(true_values, df_imputed.loc[missing_indices, target_col])

# MICE
imputer = IterativeImputer(max_iter=10, random_state=42)
df_imputed = pd.DataFrame(imputer.fit_transform(df_num_sim), columns=num_cols)
results['MICE'] = compute_rmse(true_values, df_imputed.loc[missing_indices, target_col])

df_results = pd.DataFrame(list(results.items()), columns=['Strategy', 'RMSE'])
df_results = df_results.sort_values(by='RMSE').reset_index(drop=True)

print("\tIMPUTATION PERFORMANCE COMPARISON (RMSE)")
display(df_results.style.format({"RMSE": "{:.4f}"}).background_gradient(cmap='RdYlGn_r', subset=['RMSE']))

	IMPUTATION PERFORMANCE COMPARISON (RMSE)


,Strategy,RMSE
0,MICE,13.6168
1,Mean,13.7441
2,Median,13.8629
3,KNN (k=10),13.9149
4,Mode,14.0301
5,KNN (k=5),14.1506
6,KNN (k=3),14.4296


### **3.4 Performance Comparison & Strategy Selection**

Dựa trên cơ sở lý thuyết và kết quả thực nghiệm RMSE với 10.11% dữ liệu khuyết nhân tạo, nhóm thiết lập bảng đối chiếu chiến lược:

| Chiến lược | Giả định | Bảo toàn phân phối | Bảo toàn tương quan | Chi phí tính toán |
| :--- | :--- | :--- | :--- | :--- |
| **Mean** | MCAR, Normal | Rất thấp (Tạo chóp nhọn) | Kém | Cực thấp |
| **Median** | MCAR, Skewed | Thấp | Kém | Cực thấp |
| **Mode** | MCAR | Thấp | Kém | Cực thấp |
| **k-NN** | MAR / MCAR | Cao (dùng mẫu cục bộ) | Trung bình - Cao | Cao |
| **MICE** | MAR | Rất Cao | Cao (hồi quy đa biến)| Khá Cao |

**The Best Strategy: Multiple Imputation by Chained Equations (MICE)**

**Lý giải:**
Đối với các ứng dụng khoa học dữ liệu chuyên nghiệp, MICE được lựa chọn là phương pháp luận ưu việt nhất dựa trên 3 nguyên nhân:

1. **Empirical Evidence:** Kết quả RMSE chứng minh MICE là thuật toán nội suy chính xác nhất (RMSE = 13.61). Đáng chú ý, các phương pháp gán hằng số toàn cục (Mean, Median) lại vượt trội hơn hẳn k-NN. Về mặt toán học, hàm mục tiêu RMSE luôn "ưu ái" các giá trị trung tâm (như Mean) để tối thiểu hóa bình phương sai số của một phân phối tập trung như tuổi tác. Trong khi đó, thuật toán k-NN bị nhiễu do khoảng cách Euclidean bị chi phối nặng nề bởi các biến có thang đo lớn (Curse of Dimensionality) nếu chưa qua chuẩn hóa. MICE đã vượt qua được ảnh hưởng của Mean nhờ khả năng khai thác triệt để mối quan hệ tuyến tính/phi tuyến giữa biến mục tiêu và các biến quan sát khác.
2. **Handling of Uncertainty:** Khác với Mean/Median coi giá trị nội suy là sự thật tuyệt đối, MICE lồng ghép phương sai giữa các lần điền khuyết (between-imputation variance). Điều này ngăn chặn hiện tượng mô hình bị overconfident và bảo vệ khoảng tin cậy của dữ liệu.
3. **Distributional Fidelity:** Thông qua chuỗi phương trình hồi quy, MICE tái tạo lại phân phối mà không làm biến dạng (không tạo chóp nhọn nhân tạo như Mean/Median). Điều này cực kỳ quan trọng đối với các thuật toán nhạy cảm với phân phối ở các bước sau.

**Next Step:** Trong quy trình pipeline chính thức, nhóm sẽ sử dụng thuật toán *MICE (IterativeImputer)* để xử lý triệt để các giá trị thực sự bị khuyết nguyên bản trong bộ dữ liệu Adult Census (tại các thuộc tính phân loại như `workclass` và `occupation`, MICE sẽ được cấu hình sử dụng bộ phân loại Logistic/Random Forest Classifier thay vì Regressor).

## **4. Outlier Detection and Treatment**

### **4.1 The meaning of each row**

### **4.2 The meaning of each column**

## **5. Distribution Analysis**

### **5.1. Overview of D'Agostino-Pearson Omnibus Test**

### **5.2. Hypothesis Formulation**

Tương tự các kiểm định tính chuẩn khác, giả thuyết được đặt ra là:
* $H_0$ (Giả thuyết không): Mẫu dữ liệu được rút ra từ một tổng thể có phân phối chuẩn.
* $H_1$ (Giả thuyết thay thế hay đối thuyết): Mẫu dữ liệu không tuân theo phân phối chuẩn.

### **5.3 Core Mathematical Algorithms**

### **5.5 Implementation & Distribution Classification**

### **5.6 Scaling Strategy Proposal**

#### Distributional Characteristics

#### Implications for Feature Scaling

#### Recommended Scaling Strategy

## **6. Multivariate Correlation & Multicollinearity Analysis**

### **6.1 Theoretical Foundation: Correlation Metrics**

### **6.2 Bivariate Analysis & Heatmap Visualization**

### **6.3 Multicollinearity Detection & Critical Analysis**

### **6.4 Proposed Treatments for Multicollinearity**

---